In [1]:

from typing import List
import pandas as pd
import csv
import json
import time

import os
import openai
from openai import OpenAI

In [ ]:
muj_api_klic = ""
client = OpenAI(api_key=muj_api_klic)

In [9]:
df1 = pd.read_csv('zkracovani_komplet_15_05.csv')
df1 ['odpoved'].value_counts()
df1 = df1[df1['odpoved'] == 'YES'].copy()
df1

,id,zkraceny,odpoved
0,0,"Tract, an AI proptech startup based in London,...",YES
1,1,"Automattic, the company behind WordPress.com a...",YES
2,2,Canva has announced its first-ever redundancy ...,YES
3,3,Data analytics startup WhyHive is shutting dow...,YES
4,4,"Northvolt, a Swedish battery manufacturer, has...",YES
...,...,...,...
3352,4015,"Anyvision, a technology company, has announced...",YES
3354,4017,Tuft & Needle is laying off a portion of its r...,YES
3355,4018,"Flytedesk, a Boulder-based advertising startup...",YES
3356,4019,"Inspirato, a Denver-based travel club, announc...",YES


In [10]:
from pydantic import BaseModel

class Article(BaseModel):
    id: int
    zkraceny: str      # shrnutí článku, sloupec pro klasifikaci
    odpoved: str       # indikátor (YES/NO)

class ClassifiedArticle(Article):
    main_category: str  # jeden z šesti hlavních názvů
    sub_category: str   # jedna z dvanácti subkategorií

In [17]:


# 1) Připravte si definice kategorií a subkategorií jako text:
category_definitions = """
Main Categories and Subcategories:

1. Financial Health
   a) Funding Shortage
   b) Revenue Decline

2. Strategic Realignment
   a) Product/Service Pivot
   b) Market Exit

3. Mergers & Acquisitions
   a) Tech Stack Consolidation
   b) Post-Acquisition Role Overlap

4. Technological Change
   a) Automation & AI Adoption
   b) Legacy System Decommissioning

5. Market & External Factors
   a) Economic Downturn
   b) Regulatory & Geopolitical Shifts

6. Operational Efficiency & Performance
   a) Infrastructure Optimization
   b) Performance-based Restructuring
"""

# 2) Funkce pro klasifikaci jednoho článku:
def classify_article(article: Article) -> ClassifiedArticle:
    system_prompt = (
        "You are an expert classifier of tech-industry layoff reasons. "
        "Given an article summary, assign it to one main category and one subcategory "
        "from the list exactly as defined."
    )
    user_prompt = f"""
{category_definitions}

Article summary:
\"\"\"{article.zkraceny}\"\"\"

Respond with a csv object with keys "main_category" and "sub_category", e.g.:

{{
  "main_category": "Technological Change",
  "sub_category": "Automation & AI Adoption"
}}
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
    
        temperature = 1,
        max_tokens = 100
    )
    result = json.loads(resp.choices[0].message.content)
    return ClassifiedArticle(
        id=article.id,
        zkraceny=article.zkraceny,
        odpoved=article.odpoved,
        main_category=result["main_category"],
        sub_category=result["sub_category"]
    )
def main():
    output_file = 'kategorizace_prubezne.csv'
    final_file = 'kategorizace.csv'
    processed_ids = set()
    

    if os.path.exists(output_file):
        df_prev = pd.read_csv(output_file, usecols=['id'])
        processed_ids = set(df_prev['id'].tolist())
        mode = 'a'  # append
    else:
        # Vytvoříme nový soubor s hlavičkou
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['id','zkraceny','odpoved','main_category','sub_category'])
        mode = 'a'

    results = []
    for idx, row in df1.iterrows():
        art_id = int(row['id'])
        if art_id in processed_ids:
            continue  # přeskočíme již zpracované

        art = Article(id=art_id, zkraceny=row['zkraceny'], odpoved=row['odpoved'])
        try:
            classified = classify_article(art)
        except Exception as e:
            print(f"Error for article {art_id}: {e}")
            continue

        # Uložení průběžně
        with open(output_file, mode, newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                classified.id,
                classified.zkraceny,
                classified.odpoved,
                classified.main_category,
                classified.sub_category
            ])

        processed_ids.add(art_id)
        results.append(classified.dict())
    
    
    

            

        # Pauza pro dodržení limitů
        time.sleep(5)

    # Uložení finálních výsledků
    df_out = pd.DataFrame(results)
    df_out.to_csv(final_file, index=False, encoding='utf-8')
    print(f"Finální klasifikace uložena do {final_file}")

main()


C:\Users\zpjev\AppData\Local\Temp\ipykernel_15512\762662108.py:110: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  results.append(classified.dict())


Error for article 1043: Expecting value: line 1 column 1 (char 0)
Error for article 1474: Expecting value: line 1 column 1 (char 0)
Error for article 1737: Expecting value: line 1 column 1 (char 0)
Error for article 3043: Expecting value: line 1 column 1 (char 0)
Error for article 3044: Expecting value: line 1 column 1 (char 0)
Error for article 3341: Expecting value: line 1 column 1 (char 0)
Error for article 3407: Expecting value: line 1 column 1 (char 0)
Error for article 3518: Expecting value: line 1 column 1 (char 0)
Finální klasifikace uložena do kategorizace.csv


In [10]:
df_zkouska = pd.read_csv('Veru_kategorizace_duvodu.csv')
df_zkouska [df_zkouska ['id']== 59]


,id,zkraceny,odpoved,main_category,sub_category
45,59,Workday announced on Wednesday that it will cu...,YES,Operational Efficiency & Performance,Performance-based Restructuring


In [86]:
df_novy_final

,INDEX,COMPANY,LAID_OFF,PERCENTAGE,INDUSTRY,STAGE,ANNOUCEMENT_DATE,CITY,COUNTRY,USD_RAISED_MM,JE_DUVOD,MAIN_CATEGORY,SUB_CATEGORY,SIZE_BY_STAGE
0,0,Tract,NaN,NaN,Real Estate,Unknown,2025-04-03,London,United Kingdom,NaN,YES,Market & External Factors,Economic Downturn,Unknown
1,1,Automattic,281.0,16.0,Other,Series E,2025-04-02,San Francisco,United States,986.0,YES,Operational Efficiency & Performance,Performance-based Restructuring,Medium
2,2,Canva,10.0,NaN,Consumer,Unknown,2025-04-02,Sydney,Australia,2500.0,YES,Technological Change,Automation & AI Adoption,Unknown
3,3,WhyHive,NaN,100.0,Data,Seed,2025-04-02,Melbourne,Australia,NaN,YES,Financial Health,Revenue Decline,Small
4,4,Northvolt,2800.0,62.0,Energy,Unknown,2025-03-31,Stockholm,Sweden,13800.0,YES,Financial Health,Funding Shortage,Large
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4021,4021,Service,NaN,100.0,Travel,Seed,2020-03-16,Los Angeles,United States,5.0,NaN,NaN,NaN,Small
4022,4022,HopSkipDrive,8.0,10.0,Transportation,Unknown,2020-03-13,Los Angeles,United States,45.0,YES,Financial Health,Revenue Decline,Small
4023,4023,Panda Squad,6.0,75.0,Consumer,Seed,2020-03-13,San Francisco,United States,1.0,NaN,NaN,NaN,Small
4024,4024,Tamara Mellon,20.0,40.0,Retail,Series C,2020-03-12,Los Angeles,United States,90.0,NO,NaN,NaN,Medium


In [19]:
df = pd.read_csv('kategorizace.csv')
df

,id,zkraceny,odpoved,main_category,sub_category
0,80,"Unbabel, a Portuguese technology company speci...",YES,Technological Change,Automation & AI Adoption
1,272,Toronto-based digital product sampling startup...,YES,Financial Health,Revenue Decline
2,273,Sonos has laid off approximately 100 employees...,YES,Operational Efficiency & Performance,Performance-based Restructuring
3,274,Biotech company Grail announced it will lay of...,YES,Operational Efficiency & Performance,Performance-based Restructuring
4,275,Healthtech startup DayTwo has announced its cl...,YES,Financial Health,Revenue Decline
...,...,...,...,...,...
2253,4015,"Anyvision, a technology company, has announced...",YES,Market & External Factors,Economic Downturn
2254,4017,Tuft & Needle is laying off a portion of its r...,YES,Market & External Factors,Economic Downturn
2255,4018,"Flytedesk, a Boulder-based advertising startup...",YES,Financial Health,Revenue Decline
2256,4019,"Inspirato, a Denver-based travel club, announc...",YES,Market & External Factors,Economic Downturn


In [42]:
#df_leni = pd.read_csv('Leni_LayoffsFinal.csv')
#df_veru = pd.read_csv('Veru_kategorizace_duvodu.csv')
#df_veru = df_veru.rename(columns={'id': 'INDEX'})
#df_tym = pd.merge(df_leni, df_veru, on='INDEX', how='left')
#df_tym.to_csv('layoffs_s_duvody_hackaton.csv', index=False)
df_akcie = pd.read_csv('Leni_2_nasdaq_nyse_vypocet_procent_zmeny (2).csv', sep=';')
df_akcie

,Unnamed: 0,INDEX,COMPANY,TICKER,ANNOUNCEMENT_DATE,DIFFERENCE,PRICE_DATE,OPEN,HIGH,LOW,CLOSE,STOCK_EXCHANGE,CITY,COUNTRY,INDUSTRY,LAID_OFF,PERCENTAGE_LAID_OFF,PREVIOUS_CLOSE,PCT_CHNG_FROM_ANNOUN
0,0,2808,10X Genomics,TXG,2022-08-04,-35,2022-06-30,45.680000,46.389999,44.639999,45.250000,NASDAQ,San Francisco,United States,Healthcare,100.0,8.0,42.400002,6.721694
1,1,2808,10X Genomics,TXG,2022-08-04,-34,2022-07-01,45.490002,47.639999,44.639999,46.860001,NASDAQ,San Francisco,United States,Healthcare,100.0,8.0,42.400002,10.518865
2,2,2808,10X Genomics,TXG,2022-08-04,-33,2022-07-01,45.490002,47.639999,44.639999,46.860001,NASDAQ,San Francisco,United States,Healthcare,100.0,8.0,42.400002,10.518865
3,3,2808,10X Genomics,TXG,2022-08-04,-32,2022-07-01,45.490002,47.639999,44.639999,46.860001,NASDAQ,San Francisco,United States,Healthcare,100.0,8.0,42.400002,10.518865
4,4,2808,10X Genomics,TXG,2022-08-04,-31,2022-07-01,45.490002,47.639999,44.639999,46.860001,NASDAQ,San Francisco,United States,Healthcare,100.0,8.0,42.400002,10.518865
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22309,22309,827,Sarcos,SAR,2023-11-14,11,2023-11-24,25.700001,25.870001,25.639999,25.780001,NYSE,Salt Lake City,United States,Aerospace,150.0,NaN,25.059999,2.873109
22310,22310,827,Sarcos,SAR,2023-11-14,12,2023-11-24,25.700001,25.870001,25.639999,25.780001,NYSE,Salt Lake City,United States,Aerospace,150.0,NaN,25.059999,2.873109
22311,22311,827,Sarcos,SAR,2023-11-14,13,2023-11-27,25.780001,25.780001,25.520000,25.690001,NYSE,Salt Lake City,United States,Aerospace,150.0,NaN,25.059999,2.513971
22312,22312,827,Sarcos,SAR,2023-11-14,14,2023-11-28,25.770000,25.770000,25.520000,25.639999,NYSE,Salt Lake City,United States,Aerospace,150.0,NaN,25.059999,2.314445


In [92]:
df_akcie_14 = df_akcie[df_akcie["DIFFERENCE"] == 0].copy()
df_akcie_14 = df_akcie_14[["INDEX", "DIFFERENCE", "PCT_CHNG_FROM_ANNOUN"]]


In [12]:
df_tym[['main_category', 'sub_category']].value_counts()

main_category                         sub_category                    
Market & External Factors             Economic Downturn                   745
Financial Health                      Revenue Decline                     468
                                      Funding Shortage                    374
Operational Efficiency & Performance  Performance-based Restructuring     337
Strategic Realignment                 Product/Service Pivot               290
Mergers & Acquisitions                Post-Acquisition Role Overlap        76
Operational Efficiency & Performance  Infrastructure Optimization          49
Market & External Factors             Regulatory & Geopolitical Shifts     37
Strategic Realignment                 Market Exit                          36
Technological Change                  Automation & AI Adoption             13
Strategic Realignment                 Performance-based Restructuring      12
                                      Post-Acquisition Role Overlap    

In [ ]:
df_novy_final = pd.read_csv('LAYOFFS_FINAL_17_05_15H.csv')
df_novy_merche = pd.merge(df_novy_final, df_akcie_14, on="INDEX", how="inner")
#df_novy_merche


In [64]:
df_novy_merche["MAIN_CATEGORY"].value_counts()


MAIN_CATEGORY
Financial Health                        56
Operational Efficiency & Performance    47
Market & External Factors               37
Strategic Realignment                   26
Mergers & Acquisitions                   4
Technological Change                     1
Name: count, dtype: int64

In [1]:
# uprav pro třetí hypotézu kendalltau 
#•	H0: Neexistuje závislost mezi procentem propuštěných a velikostí změny ceny akcií. 
#•	H1: Existuje závislost mezi procentem propouštěných a velikostí změny ceny akcií.
"""Tyto rozdíly jsme zjišťovali pomocí grafu v Tableau, kde jsme použili filtry na velikost, čas a důvod. Pro ověření pozorování zkusím porovnávání s Kendallovým Tau opět v Pythonu. 
(Výsledný koeficient je --- )
(konkordantní (shodný), pokud pořadí obou proměnných je stejné,
diskordantní (neshodný), pokud pořadí obou proměnných je opačné.)"""
import statistics
from scipy import stats
from scipy.stats import kendalltau
import pandas as pd
df = pd.read_csv("Leni_2_nasdaq_nyse_vypocet_procent_zmeny (2).csv", sep=";")
df.columns = df.columns.str.strip()
df.columns
data = df[['PERCENTAGE_LAID_OFF', 'PCT_CHNG_FROM_ANNOUN']].dropna()
statistic, p_value = kendalltau(data['PERCENTAGE_LAID_OFF'], data['PCT_CHNG_FROM_ANNOUN'])

print("Kendall's tau:", statistic)
print("p-hodnota:", p_value)




Kendall's tau: -0.027289960005473353
p-hodnota: 1.9251369419257317e-06


In [ ]:

from scipy.stats import kruskal
H, p = kruskal(*groups)
print("H-statistic:", H)
print("p-value:", p)


H-statistic: 3.6380159143440465
p-value: 0.45720522340752956


In [ ]:
#nepoužitelné, pokažené, zvolen jiný statistický test
# H0 procentuální zastoupení důvodů propouštění se neliší.
# H1 procentuální zastoupení důvodů propuštění se liší. 
"""Spočítám % zastoupení důvodů v oborech a pak jaký je rozdíl mezi obory"""
df_agr_transf = pd.read_csv("LAYOFFS_FINAL_17_05_15H.csv")


pivot = pd.crosstab(
    index=df_agr_transf['MAIN_CATEGORY'],
    columns=df_agr_transf['INDUSTRY'],
    normalize='columns'  # procenta uvnitř každého sloupce (odvětví)
) * 100  # převedení na procenta

pivot = pivot.round(2)

industry_counts = df_agr_transf['INDUSTRY'].value_counts()
top5 = industry_counts.head(5).index.tolist()

print(top5)


pivot_top5 = pivot[top5]

groups = [
    pivot_top5[top5].values
    for industry_counts in top5
    if len(pivot_top5[top5].values) > 1
]

from scipy.stats import kruskal
H, p = kruskal(*groups)
print("H-statistic:", H)
print("p-value:", p)


['Finance', 'Retail', 'Healthcare', 'Other', 'Transportation']
H-statistic: [0. 0. 0. 0. 0.]
p-value: [1. 1. 1. 1. 1.]


In [ ]:
# toto je fungující Chí-2 test pro čtvrtou hypotézu
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np


# H0 procentuální zastoupení důvodů propouštění se neliší.
# H1 procentuální zastoupení důvodů propuštění se liší. 
"""Spočítám % zastoupení důvodů v oborech a pak jaký je rozdíl mezi obory"""
df = pd.read_csv("LAYOFFS_FINAL_21_05 (1).csv")

top_industries = df['INDUSTRY'].value_counts().head(6).index
df = df[(df['INDUSTRY'].isin(top_industries)) & (df['INDUSTRY'] != "Other")]

raw_counts = pd.crosstab(df['INDUSTRY'], df['MAIN_CATEGORY'])
raw_counts = raw_counts.drop(columns=["Technological Change"])
print(raw_counts)
chi2, p, dof, expected = chi2_contingency(raw_counts)
print(f"P-value: {p:.4f}")

MAIN_CATEGORY   Financial Health  Market & External Factors  \
INDUSTRY                                                      
Consumer                      45                         40   
Finance                      106                        111   
Healthcare                    82                         33   
Retail                        81                         59   
Transportation                67                         44   

MAIN_CATEGORY   Mergers & Acquisitions  Operational Efficiency & Performance  \
INDUSTRY                                                                       
Consumer                             9                                    24   
Finance                              6                                    59   
Healthcare                           5                                    33   
Retail                               6                                    31   
Transportation                       4                                    19  

In [70]:
df_novy_merche.groupby("MAIN_CATEGORY")["PCT_CHNG_FROM_ANNOUN"].mean()

MAIN_CATEGORY
Financial Health                         0.019174
Market & External Factors                5.987588
Mergers & Acquisitions                 -17.569283
Operational Efficiency & Performance     2.544691
Strategic Realignment                    2.649019
Technological Change                    15.620871
Name: PCT_CHNG_FROM_ANNOUN, dtype: float64

In [77]:
df_novy_final["finantial_health"] = df_novy_final["MAIN_CATEGORY"] == "Financial Health"
df_novy_final.groupby('SIZE_BY_STAGE')["finantial_health"].mean()

SIZE_BY_STAGE
Large      0.224096
Medium     0.180269
Small      0.232698
Unknown    0.194320
Name: finantial_health, dtype: float64